# Panorama des modèles IA : coût, compute, temps, données, énergie

Exploration globale des modèles IA (tous domaines) pour comparer compute, coûts, temps d'entraînement, taille de données et indicateurs d'énergie.
Sources : `../data/ai_models/all_ai_models.csv` et `../data/ai_models/notable_ai_models.csv`.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 5)
root = Path('..') / 'data' / 'ai_models'

all_df = pd.read_csv(root / 'all_ai_models.csv')
notable_df = pd.read_csv(root / 'notable_ai_models.csv')

# Colonnes d'intérêt
num_cols = ['Parameters', 'Training compute (FLOP)', 'Training time (hours)',
            'Training compute cost (2023 USD)', 'Training dataset size (gradients)', 'Training power draw (W)']

for col in num_cols:
    if col in all_df.columns:
        all_df[col] = pd.to_numeric(all_df[col], errors='coerce')
    if col in notable_df.columns:
        notable_df[col] = pd.to_numeric(notable_df[col], errors='coerce')

# Assemblage priorisant notable pour champs énergie/coût
common_cols = list(set(all_df.columns) | set(notable_df.columns))
# On concat et garde les colonnes clés seulement
merged = pd.concat([all_df, notable_df], ignore_index=True, sort=False)
merged['Publication date'] = pd.to_datetime(merged.get('Publication date'), errors='coerce')
merged['Model accessibility'] = merged.get('Model accessibility').fillna('Unknown')
merged['Organization'] = merged.get('Organization').fillna('Unknown')
merged['Domain'] = merged.get('Domain').fillna('Unknown')

# Filtrer valeurs aberrantes (compute >1e27 FLOP considéré outlier pour lisibilité)
merged = merged[(merged['Training compute (FLOP)'].isna()) | (merged['Training compute (FLOP)'] < 1e27)]
merged.head()


In [ ]:
# Statistiques globales par accessibilité (open/closed/unknown)
access_stats = merged.groupby('Model accessibility')[['Parameters','Training compute (FLOP)','Training time (hours)','Training compute cost (2023 USD)']].median().reset_index()
access_stats

In [ ]:
# Distribution paramètres et compute
fig, axes = plt.subplots(1,2, figsize=(12,5))
sns.histplot(merged['Parameters'].dropna(), bins=50, log_scale=True, ax=axes[0], color='teal')
axes[0].set_title('Distribution des paramètres (log)')
sns.histplot(merged['Training compute (FLOP)'].dropna(), bins=50, log_scale=True, ax=axes[1], color='orange')
axes[1].set_title('Distribution du compute entraînement (log)')
plt.tight_layout(); plt.show()


In [ ]:
# Relation paramètres vs compute (log-log)
subset = merged.dropna(subset=['Parameters','Training compute (FLOP)'])
subset['log_params'] = np.log10(subset['Parameters'])
subset['log_compute'] = np.log10(subset['Training compute (FLOP)'])
ax = sns.scatterplot(data=subset, x='log_params', y='log_compute', hue=subset['Model accessibility'], alpha=0.6)
ax.set_xlabel('log10(Paramètres)')
ax.set_ylabel('log10(Compute entraînement FLOP)')
ax.set_title('Paramètres vs Compute')
plt.tight_layout(); plt.show()


In [ ]:
# Efficience algorithme : FLOP par paramètre
subset['compute_per_param'] = subset['Training compute (FLOP)'] / subset['Parameters']
ax = sns.histplot(subset['compute_per_param'].dropna(), bins=60, log_scale=True, color='purple')
ax.set_xlabel('FLOP par paramètre')
ax.set_title('Efficience (FLOP/paramètre)')
plt.tight_layout(); plt.show()


In [ ]:
# Temps vs compute : efficience infra (heures pour un volume de compute)
subset_time = merged.dropna(subset=['Training compute (FLOP)','Training time (hours)'])
subset_time['log_compute'] = np.log10(subset_time['Training compute (FLOP)'])
subset_time['log_time'] = np.log10(subset_time['Training time (hours)'])
ax = sns.scatterplot(data=subset_time, x='log_compute', y='log_time', hue=subset_time['Organization'], alpha=0.4)
ax.set_xlabel('log10(Compute FLOP)')
ax.set_ylabel('log10(Temps h)')
ax.set_title('Compute vs Temps entraînement')
plt.tight_layout(); plt.show()


In [ ]:
# Coût vs compute (quand disponible)
subset_cost = merged.dropna(subset=['Training compute (FLOP)','Training compute cost (2023 USD)'])
subset_cost['cost_per_flop'] = subset_cost['Training compute cost (2023 USD)'] / subset_cost['Training compute (FLOP)']
ax = sns.scatterplot(data=subset_cost, x='Training compute (FLOP)', y='Training compute cost (2023 USD)', alpha=0.5)
ax.set_xscale('log'); ax.set_yscale('log')
ax.set_title('Coût vs Compute')
plt.tight_layout(); plt.show()


In [ ]:
# Évolution temporelle du compute/paramètres (médiane annuelle)
ann = merged.dropna(subset=['Publication date'])
ann['year'] = ann['Publication date'].dt.year
ann_stats = ann.groupby('year')[['Parameters','Training compute (FLOP)']].median().reset_index()
fig, axes = plt.subplots(1,2, figsize=(12,4))
sns.lineplot(data=ann_stats, x='year', y='Parameters', marker='o', ax=axes[0])
axes[0].set_yscale('log'); axes[0].set_title('Médiane paramètres par année')
sns.lineplot(data=ann_stats, x='year', y='Training compute (FLOP)', marker='o', ax=axes[1])
axes[1].set_yscale('log'); axes[1].set_title('Médiane compute par année')
plt.tight_layout(); plt.show()


## Lecture rapide
- Distributions très étalées (plusieurs ordres de grandeur) pour paramètres et compute.
- La relation params→compute est super-linéaire : les gros modèles exigent des volumes de compute disproportionnés.
- Le coût vs compute est fortement corrélé mais peu renseigné : les points disponibles montrent des coûts qui explosent avec le compute.
- Les médianes annuelles progressent rapidement, illustrant la pression sur l’énergie et les coûts.
- Les ratios FLOP/paramètre varient, signe d’optimisations algorithmiques ou de stratégies de données.
